# 04. Patient-level split and shape-preserving resize

**Group 5, ICSI 535: Chest X-Ray Pneumonia project.** This is the first data item on our checklist. Every model (dense, CNN, ResNet) should be retrained on what this notebook produces.

**Two problems with our first data preparation, in plain words:**

1. **Stretched images.** X-rays come in many shapes (some wide, some tall). Resizing straight to 224 x 224 squashes them, so ribs and lungs look fatter or thinner than they are. **Fix:** add black bars to make each X-ray square first, then shrink it to 224 x 224. Nothing gets stretched.
2. **The same patient on both sides of the split.** Many patients have several X-rays. Our old split shuffled *images*, so one X-ray of a patient could land in training and another in validation. The model can then "recognise the patient" instead of learning pneumonia, which makes validation scores look better than they really are. **Fix:** read the patient number from each file name and keep all of a patient's X-rays in the same pile.

**What this notebook makes** (in `data/processed_224_pad_gray/`):

| File | What it is |
|---|---|
| `train.csv`, `validation.csv`, `test.csv` | one row per X-ray: path, label, class name, patient ID, split |
| `train.npz`, `validation.npz`, `test.npz` | the padded 224 x 224 grayscale images, **same format as our old `processed_224_gray`** (arrays `images` and `labels`) |
| `xray_data.py` | the shared loading code everyone imports (dense, CNN and ResNet) |
| `reports/` | before/after picture, split summary and the leakage numbers for the slides |

**Rules we keep:** fixed seed 42, about 80/20 train/validation from the original train + val folders, and the **original test set is not changed**.

**How to run:** fix the settings in Step 0 if needed, then *Run all*. It takes about 3 to 5 minutes. No GPU needed.

## Step 0. Settings

**Running it in our GitHub repo (like notebooks 01 to 03)?** Put this notebook in `notebooks/`. It finds `data/chest_xray` by itself.

**Running it in Google Colab?** Set `USE_GOOGLE_DRIVE = True` to keep the results in your Drive (recommended, because Colab deletes files when it resets) and set `DRIVE_PROJECT_DIR` to a folder there. If the Kaggle images aren't there, the notebook downloads them.

In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False                        # Colab only: True = save everything in your Google Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/535_project"

SEED = 42                                       # same seed as the rest of the project
VAL_FRACTION = 0.20                             # about 20% of patients go to validation

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "chest_xray"                 # Kaggle images: train/, val/, test/
OUTPUT_DIR = PROJECT_DIR / "data" / "processed_224_pad_gray"  # everything this notebook makes
REPORT_DIR = OUTPUT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Output folder: ", OUTPUT_DIR)

## Step 1. The shared code (`xray_data.py`)

The next cell saves our shared code as a small Python file and the cell after it loads it. **Teammates only need this file** (plus the outputs) to load the data the same way: copy `xray_data.py` next to your notebook and use

```python
from xray_data import load_split, make_dataset
X_train, y_train = load_split("train", PROCESSED_DIR, model="cnn")   # or model="dense"
train_ds = make_dataset("train", PROCESSED_DIR, model="resnet", shuffle=True)
```

It holds four things: the padding resize, the patient-ID reader, the loaders, and the class-weight formula.

In [ ]:
%%writefile xray_data.py
"""Shared data code for Group 5 (ICSI 535): chest X-ray pneumonia project.

Everyone loads the same images, the same splits and the same preprocessing from here,
so the dense network, the CNN and the ResNet are compared fairly.

    from xray_data import load_split, make_dataset

    X_train, y_train = load_split("train", PROCESSED_DIR, model="cnn")         # numpy arrays
    train_ds = make_dataset("train", PROCESSED_DIR, model="resnet", shuffle=True)  # tf.data pipeline

Splits are "train", "validation" and "test". Labels: 0 = NORMAL, 1 = PNEUMONIA.
"""
import re
from pathlib import Path

import numpy as np
from PIL import Image

IMAGE_SIZE = 224
SEED = 42
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
IMAGE_EXTENSIONS = {".jpeg", ".jpg", ".png", ".bmp"}


# ---------------------------------------------------------------------------
# 1. Preprocessing: grayscale, pad to a square with black bars, resize to 224 x 224
# ---------------------------------------------------------------------------
def pad_and_resize(img, size=IMAGE_SIZE):
    """Grayscale PIL image -> size x size, keeping the shape (no stretching)."""
    img = img.convert("L")
    width, height = img.size
    side = max(width, height)
    square = Image.new("L", (side, side), 0)                      # black canvas
    square.paste(img, ((side - width) // 2, (side - height) // 2))  # X-ray in the middle
    return square.resize((size, size), Image.Resampling.LANCZOS)


def stretch_resize(img, size=IMAGE_SIZE):
    """The old way (squashes the image). Kept only for before/after pictures."""
    return img.convert("L").resize((size, size), Image.Resampling.LANCZOS)


def load_image(path, size=IMAGE_SIZE):
    with Image.open(path) as img:
        return np.asarray(pad_and_resize(img, size), dtype=np.uint8)


# ---------------------------------------------------------------------------
# 2. Patient IDs from Kaggle file names
# ---------------------------------------------------------------------------
# PNEUMONIA: person1_bacteria_1.jpeg, person1_virus_6.jpeg      -> patient "person1"
# NORMAL:    IM-0115-0001.jpeg                                  -> patient "IM-0115"
#            NORMAL2-IM-0927-0001.jpeg (also ...-0001-0002)     -> patient "NORMAL2-IM-0927"
PATIENT_PATTERNS = [
    re.compile(r"^(person\d+)_(?:bacteria|virus)_\d+", re.IGNORECASE),
    re.compile(r"^((?:NORMAL2-)?IM-\d+)-\d+(?:-\d+)*$", re.IGNORECASE),
]


def patient_id_from_filename(filename):
    """Returns (patient_id, found). If no pattern matches, the file is its own patient."""
    stem = Path(filename).stem
    for pattern in PATIENT_PATTERNS:
        match = pattern.match(stem)
        if match:
            return match.group(1), True
    return stem, False


# ---------------------------------------------------------------------------
# 3. Loading the saved splits (same .npz format as data/processed_224_gray)
# ---------------------------------------------------------------------------
def load_npz(name, processed_dir):
    """Raw uint8 images (N, 224, 224) and int64 labels, exactly as saved."""
    with np.load(Path(processed_dir) / f"{name}.npz") as data:
        return data["images"], data["labels"].astype(np.int64)


def _shape_for_model(images, model):
    """images: float32 (N, 224, 224) scaled 0-1."""
    if model == "dense":
        return images                                   # (N, 224, 224), like 02_dense_baseline
    if model == "cnn":
        return images[..., np.newaxis]                  # (N, 224, 224, 1), like 03_cnn_from_scratch
    raise ValueError("model must be 'dense', 'cnn' or 'resnet'")


def load_split(name, processed_dir, model="cnn"):
    """Numpy arrays for model.fit(): pixels 0-1, shaped for 'dense' or 'cnn'.

    For the ResNet use make_dataset(..., model="resnet"): 3-channel float copies of
    every image would use several GB of memory, so the pipeline converts batch by batch.
    """
    if model == "resnet":
        raise ValueError("Use make_dataset(name, processed_dir, model='resnet') for the ResNet.")
    images, labels = load_npz(name, processed_dir)
    return _shape_for_model(images.astype(np.float32) / 255.0, model), labels


def make_dataset(name, processed_dir, model="cnn", batch_size=32, shuffle=False,
                 seed=SEED, resnet_preprocess="resnet50"):
    """tf.data pipeline of (image, label) batches.

    model="dense"  -> (224, 224), pixels 0-1
    model="cnn"    -> (224, 224, 1), pixels 0-1
    model="resnet" -> (224, 224, 3): the gray channel copied 3 times, then ImageNet
                      normalization. resnet_preprocess="resnet50" for
                      keras.applications.ResNet50, "resnet50v2" for ResNet50V2.
    """
    import tensorflow as tf

    images, labels = load_npz(name, processed_dir)
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(len(labels), seed=seed, reshuffle_each_iteration=True)

    if model == "resnet":
        if resnet_preprocess == "resnet50":
            from keras.applications.resnet50 import preprocess_input
        elif resnet_preprocess == "resnet50v2":
            from keras.applications.resnet_v2 import preprocess_input
        else:
            raise ValueError("resnet_preprocess must be 'resnet50' or 'resnet50v2'")

        def to_input(x, y):
            x = tf.image.grayscale_to_rgb(tf.cast(x, tf.float32)[..., tf.newaxis])  # 0-255, 3 channels
            return preprocess_input(x), y
    elif model in ("dense", "cnn"):
        def to_input(x, y):
            x = tf.cast(x, tf.float32) / 255.0
            return (x if model == "dense" else x[..., tf.newaxis]), y
    else:
        raise ValueError("model must be 'dense', 'cnn' or 'resnet'")

    ds = ds.batch(batch_size).map(to_input, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


def class_weights(labels):
    """Balanced class weights, the same formula 03_cnn_from_scratch uses."""
    counts = np.bincount(labels, minlength=2)
    return {i: float(len(labels) / (2 * counts[i])) for i in range(2)}

In [ ]:
import importlib, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

import xray_data
importlib.reload(xray_data)    # picks up the file we just saved, even on a re-run
from xray_data import (CLASS_NAMES, IMAGE_EXTENSIONS, IMAGE_SIZE, load_image, pad_and_resize,
                       patient_id_from_filename, stretch_resize)

random.seed(SEED)
np.random.seed(SEED)
shutil.copy("xray_data.py", OUTPUT_DIR / "xray_data.py")   # a copy to share with the group
print("Shared code ready. Image size:", IMAGE_SIZE)

## Step 2. Find the Kaggle images

Uses `data/chest_xray` if it is already there (as in our repo). Otherwise it downloads the Kaggle *Chest X-Ray Images (Pneumonia)* dataset with `kagglehub`.

In [ ]:
def has_splits(folder):
    return all((folder / split / "NORMAL").is_dir() for split in ["train", "val", "test"])

if not has_splits(RAW_DIR):
    import kagglehub   # in Colab already installed; elsewhere: pip install kagglehub
    download_root = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
    # The Kaggle zip contains the data folder more than once; use the shortest path that has train/val/test.
    candidates = sorted(
        (p for p in [download_root, *download_root.rglob("*")]
         if p.is_dir() and "__MACOSX" not in str(p) and has_splits(p)),
        key=lambda p: len(str(p)),
    )
    RAW_DIR = candidates[0]

print("Using images from:", RAW_DIR)

## Step 3. List every X-ray and read its patient ID

Kaggle file names tell us the patient:

| Class | Example file name | Patient ID we read |
|---|---|---|
| PNEUMONIA | `person1_bacteria_1.jpeg`, `person1_virus_6.jpeg` | `person1` |
| NORMAL | `IM-0115-0001.jpeg` | `IM-0115` |
| NORMAL | `NORMAL2-IM-0927-0001.jpeg` | `NORMAL2-IM-0927` |

Paths are saved **relative to the image folder** (for example `train/NORMAL/IM-0115-0001.jpeg`), so the CSVs work on anyone's computer.

In [ ]:
rows = []
for original_split in ["train", "val", "test"]:
    for label, class_name in enumerate(CLASS_NAMES):
        for path in sorted((RAW_DIR / original_split / class_name).iterdir()):
            if not (path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS):
                continue
            patient_id, found = patient_id_from_filename(path.name)
            with Image.open(path) as img:
                width, height = img.size
            rows.append({
                "path": path.relative_to(RAW_DIR).as_posix(),
                "label": label,
                "class_name": class_name,
                "patient_id": patient_id,
                "original_split": original_split,
                "patient_id_found": found,
                "width": width,
                "height": height,
            })

all_df = pd.DataFrame(rows)
print(f"{len(all_df)} images, {all_df['patient_id'].nunique()} patients")
print(all_df.groupby(["original_split", "class_name"]).size().unstack())
print("Expected from our notes: train 1341 / 3875, test 234 / 390, val 8 / 8 (total 5856)")

### Spot-check the patient IDs

Look at the random examples below: does the patient ID match the start of the file name? Also check how many files did **not** match a pattern (each of those is treated as its own patient, which is the safe default only if the number is small).

In [ ]:
sample = all_df.sample(12, random_state=SEED)[["path", "class_name", "patient_id"]]
display(sample.reset_index(drop=True))

unmatched = all_df[~all_df["patient_id_found"]]
print(f"Files whose name didn't match a pattern: {len(unmatched)}")
if len(unmatched):
    display(unmatched[["path", "patient_id"]].head(20))

per_patient = all_df.groupby("patient_id").size()
print(f"\nX-rays per patient: median {per_patient.median():.0f}, max {per_patient.max()}")
print(f"Patients with more than one X-ray: {(per_patient > 1).sum()} "
      f"(they own {per_patient[per_patient > 1].sum()} of {len(all_df)} images)")
print("\nPatients by class:")
print(all_df.groupby("class_name")["patient_id"].nunique())
print("\nPatients that appear in both classes:",
      (all_df.groupby("patient_id")["class_name"].nunique() > 1).sum())

## Step 4. How much did the old split mix patients?

We redo our first split exactly as notebook 01 did it (original train + val combined, image-level stratified 80/20, `random_state=42`) and count patients that ended up on **both** sides.

**Honest caveat:** notebook 01 listed files in the order the computer returns them, so on a different computer a few images may land on the other side. The numbers here are therefore a close estimate of our old split, not an exact copy.

In [ ]:
# Same steps as notebook 01 (including its unsorted file order)
old_rows = []
for split in ["train", "val"]:
    for label, class_name in enumerate(CLASS_NAMES):
        for path in (RAW_DIR / split / class_name).iterdir():
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                old_rows.append({"path": path.relative_to(RAW_DIR).as_posix(), "label": label})
old_df = pd.DataFrame(old_rows).merge(all_df[["path", "patient_id"]], on="path")
old_train, old_val = train_test_split(old_df, test_size=0.20, random_state=42, stratify=old_df["label"])

shared = set(old_train["patient_id"]) & set(old_val["patient_id"])
val_images_leaked = old_val["patient_id"].isin(shared).sum()
old_leak = {
    "old_train_images": len(old_train),
    "old_val_images": len(old_val),
    "patients_in_both": len(shared),
    "val_patients": old_val["patient_id"].nunique(),
    "val_images_with_same_patient_in_train": int(val_images_leaked),
}
print(f"Old split: {len(old_train)} train / {len(old_val)} validation images")
print(f"Patients in BOTH train and validation: {len(shared)} of {old_val['patient_id'].nunique()} validation patients")
print(f"Validation images whose patient was also in training: {val_images_leaked} "
      f"({val_images_leaked / len(old_val):.1%})")

## Step 5. The new split: grouped by patient

We use scikit-learn's `StratifiedGroupKFold`. **Grouped** means all X-rays of one patient stay together. **Stratified** means train and validation keep roughly the same NORMAL / PNEUMONIA mix. We cut the patients into 5 equal parts and use one part (about 20%) as validation.

The **test set stays exactly as Kaggle gave it** (624 images). We only check whether any test patient ID also shows up in train or validation, and report it.

In [ ]:
pool = all_df[all_df["original_split"].isin(["train", "val"])].reset_index(drop=True)
test_df = all_df[all_df["original_split"] == "test"].copy()

n_folds = round(1 / VAL_FRACTION)
splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
train_idx, val_idx = next(splitter.split(pool, pool["label"], groups=pool["patient_id"]))

train_df = pool.iloc[train_idx].copy()
val_df = pool.iloc[val_idx].copy()
train_df["split"], val_df["split"], test_df["split"] = "train", "validation", "test"

# The key check: no patient on both sides
new_shared = set(train_df["patient_id"]) & set(val_df["patient_id"])
assert not new_shared, f"{len(new_shared)} patients are in both train and validation!"
assert len(train_df) + len(val_df) == len(pool)
print("Check passed: no patient is in both train and validation.\n")

summary = pd.DataFrame([
    {"split": name, "images": len(df), "patients": df["patient_id"].nunique(),
     "NORMAL": int((df["label"] == 0).sum()), "PNEUMONIA": int((df["label"] == 1).sum()),
     "% NORMAL": round(100 * (df["label"] == 0).mean(), 1)}
    for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]
])
display(summary)
print("Old split for comparison: train 4185 (1079 / 3106), validation 1047 (270 / 777), test 624 (234 / 390)")

test_overlap = set(test_df["patient_id"]) & set(pool["patient_id"])
test_images_overlap = test_df["patient_id"].isin(test_overlap).sum()
print(f"\nTest patients whose ID also appears in train/validation: {len(test_overlap)} "
      f"({test_images_overlap} of {len(test_df)} test images). The test set was NOT changed.")
if test_overlap:
    print("Examples:", sorted(test_overlap)[:10])

## Step 6. Before and after: stretched vs. padded

Left: the original X-ray. Middle: the old way, squashed to a square. Right: the new way, black bars added and then shrunk, so the body keeps its true shape. We show the widest and the tallest images, where the difference is easiest to see.

In [ ]:
all_df["aspect"] = all_df["width"] / all_df["height"]
print(f"Width / height ranges from {all_df['aspect'].min():.2f} to {all_df['aspect'].max():.2f} "
      f"(median {all_df['aspect'].median():.2f}); 1.00 would be a perfect square.")

examples = [all_df.loc[all_df["aspect"].idxmax()], all_df.loc[all_df["aspect"].idxmin()],
            all_df[all_df["class_name"] == "PNEUMONIA"].sample(1, random_state=SEED).iloc[0]]

fig, axes = plt.subplots(len(examples), 3, figsize=(11, 3.6 * len(examples)))
for row_axes, ex in zip(axes, examples):
    with Image.open(RAW_DIR / ex["path"]) as img:
        original = img.convert("L")
        panels = [(original, f"{ex['class_name']}: original {ex['width']} x {ex['height']}"),
                  (stretch_resize(original), "Old: stretched to 224 x 224"),
                  (pad_and_resize(original), "New: padded, then 224 x 224")]
    for ax, (picture, title) in zip(row_axes, panels):
        ax.imshow(picture, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
plt.tight_layout()
plt.savefig(REPORT_DIR / "resize_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7. Save the CSVs and the padded images

Each CSV row is one X-ray. **Row order matches the `.npz` file**: row 0 of `train.csv` is image 0 of `train.npz` (the `npz_index` column says so too). Columns:

`path` (relative to the image folder), `label` (0 = NORMAL, 1 = PNEUMONIA), `class_name`, `patient_id`, `split`, `original_split`, `npz_index`, `width`, `height`.

The `.npz` files use the **same names and format as our old `data/processed_224_gray`** (`images` uint8 of shape N x 224 x 224, `labels`), so notebooks 02 and 03 only need one line changed.

In [ ]:
COLUMNS = ["path", "label", "class_name", "patient_id", "split", "original_split", "npz_index", "width", "height"]

splits = {}
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    df = df.sort_values("path").reset_index(drop=True)
    df["npz_index"] = np.arange(len(df))
    df[COLUMNS].to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    splits[name] = df

for name, df in splits.items():
    images = np.empty((len(df), IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    for i, rel_path in enumerate(tqdm(df["path"], desc=f"Padding {name}")):
        images[i] = load_image(RAW_DIR / rel_path)
    np.savez_compressed(OUTPUT_DIR / f"{name}.npz", images=images, labels=df["label"].to_numpy(np.int64))
    print(f"{name}: {images.shape} saved, {(OUTPUT_DIR / f'{name}.npz').stat().st_size / 1024**2:.1f} MB")

summary.to_csv(REPORT_DIR / "split_summary.csv", index=False)
pd.DataFrame([{**old_leak,
               "new_patients_in_both": 0,
               "test_patient_ids_also_in_train_or_val": len(test_overlap),
               "test_images_with_those_ids": int(test_images_overlap)}]).to_csv(
    REPORT_DIR / "leakage_report.csv", index=False)
print("\nSaved to", OUTPUT_DIR)

## Step 8. Check that the shared loader works for all three models

We load each split the way each model needs it and print the shapes. We also re-process one CSV row by hand and confirm it equals the matching `.npz` image, which proves the CSV and the `.npz` line up.

In [ ]:
from xray_data import class_weights, load_split, make_dataset

X_dense, y = load_split("train", OUTPUT_DIR, model="dense")
X_cnn, _ = load_split("validation", OUTPUT_DIR, model="cnn")
print("dense input:", X_dense.shape, "| cnn input:", X_cnn.shape, "| pixel range", X_cnn.min(), "to", X_cnn.max())
print("class weights for the CNN:", class_weights(y))
del X_dense, X_cnn

try:
    resnet_ds = make_dataset("validation", OUTPUT_DIR, model="resnet", batch_size=8)
    xb, yb = next(iter(resnet_ds))
    print("resnet batch:", tuple(xb.shape), "labels", yb.numpy())
except ImportError:
    print("TensorFlow not installed here; the ResNet pipeline needs it (Colab has it).")

row = splits["train"].iloc[0]
raw_images, raw_labels = xray_data.load_npz("train", OUTPUT_DIR)
assert np.array_equal(load_image(RAW_DIR / row["path"]), raw_images[row["npz_index"]])
assert raw_labels[row["npz_index"]] == row["label"]
print("CSV and .npz line up.")

## Step 9. What to tell the team, and what to put on the slides

**For teammates (notebooks 02 and 03):** change one line,

```python
PROCESSED_DIR = PROJECT_DIR / "data" / "processed_224_pad_gray"   # was "processed_224_gray"
```

and rerun. Everything else stays the same. **For the ResNet:** copy `xray_data.py` next to the notebook and use `make_dataset("train", PROCESSED_DIR, model="resnet", shuffle=True)`. It gives 3-channel images with ImageNet normalization for `keras.applications.ResNet50` (use `resnet_preprocess="resnet50v2"` for ResNet50V2).

**Expect slightly different numbers.** Validation scores may drop a little after this fix. That is good news for honesty: the old validation set contained patients the model had already seen.

**The second dataset** (RSNA or other) can reuse `pad_and_resize` from `xray_data.py`, so both datasets get the same preprocessing.

The cell below prints the numbers for the slide.

In [ ]:
print("SLIDE: data fixes")
print(f"- Resize: X-rays ranged from {all_df['aspect'].min():.2f} to {all_df['aspect'].max():.2f} width/height; "
      "now padded to square before resizing (no stretching).")
print(f"- Old image-level split: {old_leak['patients_in_both']} patients had X-rays in both train and validation; "
      f"{old_leak['val_images_with_same_patient_in_train']} of {old_leak['old_val_images']} validation images "
      f"({old_leak['val_images_with_same_patient_in_train'] / old_leak['old_val_images']:.0%}) had a 'sibling' in training.")
print(f"- New patient-level split: 0 shared patients. "
      + ", ".join(f"{r.split} {r.images} images / {r.patients} patients" for r in summary.itertuples()))
print(f"- Test set unchanged; {len(test_overlap)} test patient IDs also appear in train/validation "
      "(the same number may not mean the same person across Kaggle's original folders).")